In [ ]:
from pathlib import Path
import sys

_here = Path.cwd().resolve()
_candidates = (_here, *_here.parents)
REPO_ROOT = next((candidate for candidate in _candidates if (candidate / "m33_pipeline").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root from {Path.cwd()}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from m33_pipeline.notebook_setup import prepare_notebook

REPO_ROOT = prepare_notebook(REPO_ROOT)
print(f"Notebook working directory set to: {REPO_ROOT}")

In [1]:
flux_method = 'summed_map'
dig_mode = 'no_dig'

from m33_pipeline.config import get_derived_config
from m33_pipeline.derived import (
    add_clustering_metrics,
    add_electron_density,
    add_logU_KK04,
    add_metallicity_columns,
    add_metallicity_error_columns,
    add_boundary_source_flags,
    add_primary_overlap_flags,
    add_symmetry_class,
    merge_field_flux_catalogs,
    write_clustering_outputs,
    write_combined_catalog,
    write_derived_stage_catalog,
    write_total_flux_catalog,
)
from m33_pipeline.validate import validate_total_catalog


# Merge per-field flux catalogs


In [2]:
derived_config = get_derived_config()
all_catalog = merge_field_flux_catalogs(method=flux_method, dig_mode=dig_mode)
all_catalog = add_primary_overlap_flags(all_catalog, match_radius_arcsec=1.0)
all_catalog = add_boundary_source_flags(all_catalog, max_zoi_pc=100)
n_overlap_groups = int(all_catalog['is_duplicate_overlap'].fillna(False).groupby(all_catalog['duplicate_group_id']).any().sum()) if 'duplicate_group_id' in all_catalog.columns else 0
n_duplicate_rows = int(all_catalog['is_duplicate_overlap'].fillna(False).sum()) if 'is_duplicate_overlap' in all_catalog.columns else 0
n_non_primary = int((~all_catalog['primary'].fillna(True)).sum()) if 'primary' in all_catalog.columns else 0
n_wr_flagged = int(all_catalog['has_wr_in_boundary'].fillna(False).sum()) if 'has_wr_in_boundary' in all_catalog.columns else 0
n_snr_flagged = int(all_catalog['has_snr_in_boundary'].fillna(False).sum()) if 'has_snr_in_boundary' in all_catalog.columns else 0
print(f"Overlap duplicate groups: {n_overlap_groups}")
print(f"Rows involved in overlap duplicates: {n_duplicate_rows}")
print(f"Rows flagged as non-primary: {n_non_primary}")
print(f"Regions containing WR stars: {n_wr_flagged}")
print(f"Regions containing SNRs: {n_snr_flagged}")
if n_overlap_groups > 0:
    dup_preview_cols = [c for c in ['field', 'region_id', 'duplicate_group_id', 'primary', 'primary_rank', 'primary_region_id', 'duplicate_score_sum_snr'] if c in all_catalog.columns]
    print(all_catalog.loc[all_catalog['is_duplicate_overlap'], dup_preview_cols].sort_values(['duplicate_group_id', 'primary_rank']).head(20).to_string(index=False))
output_path = write_total_flux_catalog(all_catalog, method=flux_method, dig_mode=dig_mode)
print("Flux method:", flux_method)
print("DIG mode:", dig_mode)
print("Combined catalog shape:", all_catalog.shape)
print("Saved combined catalog to:", output_path)
# validate_total_catalog(all_catalog)


Combined catalog shape: (6505, 150)
Saved combined catalog to: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/total_flux_catalog.csv


# Add ionization parameter


In [3]:
cat = add_logU_KK04(all_catalog.copy(), n_mc=derived_config.logu_n_mc, seed=123, metallicity_cal="M13_O3N2")
derived_output_path = write_derived_stage_catalog(cat, "ionization_parameter", method=flux_method, dig_mode=dig_mode)
print("Number of columns:", len(cat.columns))
print("Saved combined catalog with ionization parameter:", derived_output_path)


Number of columns: 165
Saved combined catalog with ionization parameter: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/total_flux_catalog_with_derived.csv


# Add electron density


In [4]:
df = add_electron_density(cat.copy(), n_mc=derived_config.density_n_mc)
derived_output_path = write_derived_stage_catalog(df, "density", method=flux_method, dig_mode=dig_mode)
print("Number of columns:", len(df.columns))
print("Saved combined catalog with electron densities:", derived_output_path)


Number of columns: 171
Saved combined catalog with electron densities: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/total_flux_catalog_with_derived.csv


# Add symmetry classification


In [5]:
df = add_symmetry_class(df.copy())
derived_output_path = write_derived_stage_catalog(df, "other_derived", method=flux_method, dig_mode=dig_mode)
print("Symmetry classification counts:")
print(df["symmetry_class"].value_counts())
print("Number of columns:", len(df.columns))
print("Saved combined catalog with other derived properties:", derived_output_path)


Symmetry classification counts:
symmetry_class
asymmetric    5362
symmetric     1121
unknown         22
Name: count, dtype: int64
Number of columns: 172
Saved combined catalog with symmetry classification: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/total_flux_catalog_with_derived_and_symmetry.csv


# Add metallicity calibrations


In [6]:
df = add_metallicity_columns(df.copy())
df = add_metallicity_error_columns(df.copy(), n_mc=derived_config.metallicity_n_mc, seed=123)
derived_output_path = write_derived_stage_catalog(df, "metallicity", method=flux_method, dig_mode=dig_mode)
combined_output_path = write_combined_catalog(df, method=flux_method, dig_mode=dig_mode)
print("Number of columns:", len(df.columns))
print("Saved combined catalog with metallicities:", derived_output_path)
print("Saved final combined catalog:", combined_output_path)


/Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/m33_pipeline/derived.py:384: RuntimeWarning: divide by zero encountered in divide
  out["Z_R_Pilyugin2016_highN2"] = 8.589 + 0.022 * np.log10(r3 / r2) + 0.399 * np.log10(n2) + (-0.137 + 0.164 * np.log10(r3 / r2) + 0.589 * np.log10(n2)) * np.log10(r2)
/Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/m33_pipeline/derived.py:384: RuntimeWarning: invalid value encountered in log10
  out["Z_R_Pilyugin2016_highN2"] = 8.589 + 0.022 * np.log10(r3 / r2) + 0.399 * np.log10(n2) + (-0.137 + 0.164 * np.log10(r3 / r2) + 0.589 * np.log10(n2)) * np.log10(r2)
/Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/m33_pipeline/derived.py:384: RuntimeWarning: divide by zero encountered in log10
  out["Z_R_Pilyugin2016_highN2"] = 8.589 + 0.022 * np.log10(r3 / r2) + 0.399 * np.log10(n2) + (-0.137 + 0.164 * np.log10(r3 / r2) + 0.589 * np.log10(n2)) * np.log10(r2)
/Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/m33_pipeline/derived.py:384: RuntimeWarning: invalid value enc

Number of columns: 224
Saved combined catalog with metallicities: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/total_flux_catalog_with_derived_and_metallicities.csv


# Add deprojected clustering metrics


In [7]:
clustered_df, global_stats, ripley_df, pcf_df = add_clustering_metrics(df.copy())
outputs = write_clustering_outputs(clustered_df, global_stats, ripley_df, pcf_df, method=flux_method, dig_mode=dig_mode)
print("Saved catalog:", outputs["catalog"])
print("Saved global stats:", outputs["global"])
print("Saved Ripley profile:", outputs["ripley"])
print("Saved pair-correlation profile:", outputs["pcf"])


Saved catalog: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/total_flux_catalog_with_deprojected_clustering_metrics.csv
Saved global stats: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/clustering_global_statistics.csv
Saved Ripley profile: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/clustering_ripley_profile.csv
Saved pair-correlation profile: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/clustering_pair_correlation_profile.csv
